In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df = pd.read_parquet("data/matryoshka_total/results.parquet")
flat = df.reset_index()

In [ ]:
# Best F1 and MCC per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best_f1 = group.loc[group["f1_score"].idxmax()]
    best_mcc = group.loc[group["mcc"].idxmax()]
    print(
        f"{bench:25s} "
        f"F1={best_f1['f1_score']:.4f} (k={best_f1['k']}, {best_f1['widths']})  "
        f"MCC={best_mcc['mcc']:.4f} (k={best_mcc['k']}, {best_mcc['widths']})"
    )

In [ ]:
# Full results table sorted by F1
flat.sort_values("f1_score", ascending=False)[
    [
        "benchmark",
        "k",
        "widths",
        "sae_l0",
        "true_l0",
        "precision",
        "recall",
        "f1_score",
        "mcc",
        "explained_variance",
        "dead_latents",
    ]
].head(12)

## F1 & MCC vs k across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "k", "widths"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
).sort_values(["benchmark", "widths", "metric", "k"])

fig = px.line(
    melted,
    x="k",
    y="score",
    color="benchmark",
    facet_col="metric",
    facet_row="widths",
    category_orders={
        "metric": ["f1_score", "mcc"],
        "widths": ["2-level", "3-level", "4-level"],
    },
    markers=True,
    labels={
        "k": "k (Matryoshka BatchTopK)",
        "score": "Score",
        "benchmark": "Benchmark",
        "widths": "Nesting Depth",
    },
    title="F1 & MCC vs k by Distribution (Matryoshka BatchTopK)",
    height=800,
    width=1100,
)
fig.show()

## Precision & Recall vs k across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "k", "widths"],
    value_vars=["precision", "recall"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="k",
    y="score",
    color="benchmark",
    facet_col="metric",
    facet_row="widths",
    category_orders={
        "metric": ["precision", "recall"],
        "widths": ["2-level", "3-level", "4-level"],
    },
    markers=True,
    labels={
        "k": "k (Matryoshka BatchTopK)",
        "score": "Score",
        "benchmark": "Benchmark",
        "widths": "Nesting Depth",
    },
    title="Precision & Recall vs k by Distribution (Matryoshka BatchTopK)",
    height=800,
    width=1100,
)
fig.show()

## F1 & MCC heatmap: benchmark × k (per nesting depth)

In [ ]:
width_levels = ["2-level", "3-level", "4-level"]
metrics = ["f1_score", "mcc"]

fig = make_subplots(
    rows=len(width_levels),
    cols=len(metrics),
    subplot_titles=[
        f"{m.replace('_', ' ').title()} — {w}" for w in width_levels for m in metrics
    ],
)

for row_i, w in enumerate(width_levels):
    sub = flat[flat["widths"] == w]
    for col_i, metric in enumerate(metrics):
        pivot = sub.pivot_table(values=metric, index="benchmark", columns="k")
        fig.add_trace(
            go.Heatmap(
                z=pivot.values,
                x=[str(c) for c in pivot.columns],
                y=pivot.index.tolist(),
                colorscale="Viridis",
                showscale=(col_i == len(metrics) - 1 and row_i == 0),
                text=pivot.values.round(3),
                texttemplate="%{text}",
                zmin=0,
                zmax=1,
            ),
            row=row_i + 1,
            col=col_i + 1,
        )

fig.update_layout(
    height=900,
    width=1100,
    title_text="Benchmark × k (Matryoshka BatchTopK) — rows: nesting depth",
)
fig.update_xaxes(title_text="k")
fig.show()

## F1 & MCC heatmap: benchmark × widths (per k)

In [ ]:
k_values = sorted(flat["k"].unique())

fig = make_subplots(
    rows=len(k_values),
    cols=len(metrics),
    subplot_titles=[
        f"{m.replace('_', ' ').title()} — k={k}" for k in k_values for m in metrics
    ],
)

for row_i, k in enumerate(k_values):
    sub = flat[flat["k"] == k]
    for col_i, metric in enumerate(metrics):
        pivot = sub.pivot_table(values=metric, index="benchmark", columns="widths")[
            width_levels
        ]
        fig.add_trace(
            go.Heatmap(
                z=pivot.values,
                x=pivot.columns.tolist(),
                y=pivot.index.tolist(),
                colorscale="Viridis",
                showscale=(col_i == len(metrics) - 1 and row_i == 0),
                text=pivot.values.round(3),
                texttemplate="%{text}",
                zmin=0,
                zmax=1,
            ),
            row=row_i + 1,
            col=col_i + 1,
        )

fig.update_layout(
    height=1200,
    width=1100,
    title_text="Benchmark × Nesting Depth (Matryoshka BatchTopK) — rows: k",
)
fig.show()

## sae_l0 vs true_l0 across distributions

In [ ]:
fig = px.scatter(
    flat,
    x="true_l0",
    y="sae_l0",
    color="benchmark",
    symbol="widths",
    hover_data=["k", "f1_score", "mcc"],
    labels={
        "true_l0": "True L0",
        "sae_l0": "SAE L0",
        "benchmark": "Benchmark",
        "widths": "Nesting Depth",
    },
    title="SAE L0 vs True L0 (each point = one config)",
    height=500,
    width=900,
)
max_val = max(flat["true_l0"].max(), flat["sae_l0"].max())
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(dash="dash", color="gray"),
)
fig.show()

## Dead latents vs k

In [ ]:
fig = px.line(
    flat,
    x="k",
    y="dead_latents",
    color="benchmark",
    facet_col="widths",
    category_orders={"widths": width_levels},
    markers=True,
    labels={
        "k": "k (Matryoshka BatchTopK)",
        "dead_latents": "Dead Latents",
        "benchmark": "Benchmark",
        "widths": "Nesting Depth",
    },
    title="Dead Latents vs k by Distribution (Matryoshka BatchTopK)",
    height=450,
    width=1100,
)
fig.show()